# 01 — Synthetic UPI Data: Exploratory Data Analysis

This notebook explores the synthetic UPI transaction dataset generated by `SyntheticUPIGenerator`.
We analyze user demographics, transaction patterns, temporal trends, and fraud baselines.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data.synthetic.generator import SyntheticUPIGenerator

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
gen = SyntheticUPIGenerator(n_users=10000, n_merchants=1000, n_transactions=50000, fraud_ratio=0.05, seed=42)
df = gen.generate()
print(f'Generated {len(df)} transactions')
print(f'Fraud rate: {df["is_fraud"].mean()*100:.2f}%')
df.head()

## 1. User Demographics

In [ ]:
users_df = pd.DataFrame.from_dict(gen.users, orient='index')
print(f'Unique users: {len(users_df)}')
users_df[['age', 'income_tier', 'city_tier', 'credit_score', 'account_age_days']].describe()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].hist(users_df['age'], bins=30, color='steelblue', edgecolor='white')
axes[0,0].set_title('Age Distribution')
axes[0,0].set_xlabel('Age')

axes[0,1].hist(users_df['credit_score'], bins=30, color='coral', edgecolor='white')
axes[0,1].set_title('Credit Score Distribution')
axes[0,1].set_xlabel('Credit Score')

users_df['income_tier'].value_counts().sort_index().plot(kind='bar', ax=axes[1,0], color='seagreen')
axes[1,0].set_title('Income Tier Distribution')
axes[1,0].set_xlabel('Income Tier')

users_df['city_tier'].value_counts().sort_index().plot(kind='bar', ax=axes[1,1], color='purple')
axes[1,1].set_title('City Tier Distribution')
axes[1,1].set_xlabel('City Tier')

plt.tight_layout()
plt.show()

## 2. Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['amount'], bins=100, color='steelblue', edgecolor='white')
axes[0].set_title('Transaction Amount Distribution (Linear)')
axes[0].set_xlabel('Amount (INR)')

axes[1].hist(np.log10(df['amount'] + 1), bins=100, color='coral', edgecolor='white')
axes[1].set_title('Transaction Amount Distribution (Log10)')
axes[1].set_xlabel('log10(Amount)')

plt.tight_layout()
plt.show()

print(f'Amount stats:')
print(f'  Mean:   INR {df["amount"].mean():,.2f}')
print(f'  Median: INR {df["amount"].median():,.2f}')
print(f'  Std:    INR {df["amount"].std():,.2f}')
print(f'  Min:    INR {df["amount"].min():,.2f}')
print(f'  Max:    INR {df["amount"].max():,.2f}')

## 3. Temporal Patterns

In [ ]:
df['hour'] = df['timestamp'].dt.hour
df['day'] = df['timestamp'].dt.day
df['weekday'] = df['timestamp'].dt.weekday

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

hourly = df.groupby('hour').size()
axes[0].bar(hourly.index, hourly.values, color='steelblue')
axes[0].set_title('Transaction Volume by Hour')
axes[0].set_xlabel('Hour of Day')

daily = df.groupby('day').size()
axes[1].plot(daily.index, daily.values, marker='o', color='coral')
axes[1].set_title('Transaction Volume by Day of Month')
axes[1].set_xlabel('Day of Month')
axes[1].axvline(5, color='gray', linestyle='--', alpha=0.5, label='Salary period end')
axes[1].legend()

weekday_counts = df.groupby('weekday').size()
labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[2].bar(labels, weekday_counts.values, color='seagreen')
axes[2].set_title('Transaction Volume by Weekday')

plt.tight_layout()
plt.show()

## 4. Salary-Day Effect

The 1st-5th of the month should show ~40% higher transaction volume.

In [ ]:
df['is_salary_period'] = df['day'] <= 5
salary_avg = df[df['is_salary_period']]['amount'].mean()
normal_avg = df[~df['is_salary_period']]['amount'].mean()
print(f'Salary period avg amount: INR {salary_avg:,.2f}')
print(f'Normal period avg amount: INR {normal_avg:,.2f}')
print(f'Ratio: {salary_avg/normal_avg:.2f}x')

## 5. MCC Category Distribution

In [ ]:
mcc_counts = df['mcc_code'].value_counts()
plt.figure(figsize=(12, 5))
mcc_counts.plot(kind='bar', color='teal')
plt.title('Transaction Volume by MCC Category')
plt.xlabel('MCC Code')
plt.xticks(rotation=45)
plt.show()

## 6. Transaction Types

In [ ]:
txn_type_counts = df['txn_type'].value_counts()
plt.figure(figsize=(8, 6))
plt.pie(txn_type_counts.values, labels=txn_type_counts.index, autopct='%1.1f%%', startangle=90)
plt.title('Transaction Type Distribution')
plt.show()

## 7. Fraud Baseline

Examining fraud distribution across patterns.

In [ ]:
fraud_df = df[df['is_fraud']]
print(f'Total fraud transactions: {len(fraud_df)}')
print(f'\nFraud pattern breakdown:')
print(fraud_df['fraud_pattern'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fraud_df['fraud_pattern'].value_counts().plot(kind='bar', ax=axes[0], color=['red', 'orange', 'purple', 'blue'])
axes[0].set_title('Fraud Pattern Distribution')
axes[0].set_xlabel('Pattern')

axes[1].scatter(df['amount'], df['is_fraud'] + np.random.uniform(-0.02, 0.02, len(df)), alpha=0.1, s=1)
axes[1].set_title('Fraud Label vs Amount')
axes[1].set_xlabel('Amount (INR)')
axes[1].set_ylabel('Is Fraud')

plt.tight_layout()
plt.show()

## 8. Correlation Analysis

In [ ]:
numeric_df = df[['amount', 'is_fraud', 'hour', 'day']].copy()
corr = numeric_df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, square=True)
plt.title('Feature Correlation Matrix')
plt.show()

## Key Observations

1. Age distribution follows a truncated normal with mean ~32 years
2. Credit scores cluster around 720 with realistic variance
3. Transaction amounts follow a log-normal distribution
4. Diurnal pattern shows clear bimodal peaks (lunch and evening)
5. Salary-day effect visible in first 5 days of month
6. Fraud patterns are balanced across the 4 injected types